In [25]:
!pip install Audiopy_ML

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score, mean_squared_error
from Audiopy_ML import autoaudio
import glob
from tqdm import tqdm

In [27]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [29]:
class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.spect_dbs[idx], self.labels[idx]

In [30]:
train_dataset = glob.glob('/kaggle/input/dcase-aml/dev_data/dev_data/slider/train/*')
test_dataset = glob.glob('/kaggle/input/dcase-aml/dev_data/dev_data/slider/test/*')

In [ ]:
def feature_extractor(file):
    def mfcc_extractor(file):
        data, sr = librosa.load(file) 
        mfccs_features = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=128)
        mfccs_scaled_features = np.mean(mfccs_features.T,axis=0)
        return mfccs_scaled_features
    def zero_extractor(file):
        data, sr = librosa.load(file)
        zeros=librosa.feature.zero_crossing_rate(data,frame_length=2048, hop_length=512, center=True)
        zeros_scaled_features= np.mean(zeros.T,axis=0)
        return zeros_scaled_features
    def rms_extractor(file):
        data, sr = librosa.load(file)    
        rms=librosa.feature.rms(y=data)
        rms_scaled_features= np.mean(rms.T,axis=0)
        return rms_scaled_features
    def spectral_centroid_extractor(file):
        data, sr = librosa.load(file)
        sc=librosa.feature.spectral_centroid(y=data,sr=sr)
        sc_scaled_features= np.mean(sc.T,axis=0)
        return sc_scaled_features
    def spectral_bandwidth_extractor(file):
        data, sr = librosa.load(file)
        sb=librosa.feature.spectral_bandwidth(y=data,sr=sr)
        sb_scaled_features= np.mean(sb.T,axis=0)
        return sb_scaled_features
    def spectral_contrast_extractor(file):
        data, sr = librosa.load(file)
        sco=librosa.feature.spectral_contrast(y=data,sr=sr)
        sco_scaled_features= np.mean(sco.T,axis=0)
        return sco_scaled_features
    def polynomial_extractor(file):
        data, sr = librosa.load(file)
        poly=librosa.feature.poly_features(y=data,sr=sr,order=2)
        poly_scaled_features= np.mean(poly.T,axis=0)
        return poly_scaled_features
    mfcc_features=[]
    zero_features=[]
    rms_features=[]
    sc_features=[]
    sb_features=[]
    sco_features=[]
    poly_features=[]
    labels = []
    for i in tqdm(file):
        label = 0 if i.split('/')[-1].startswith('n') else 1
        labels.append(label)
        
        mfcc=mfcc_extractor(i)
        mfcc_features.append(mfcc)

        zero=zero_extractor(i)
        zero_features.append(zero)

        rms=rms_extractor(i)
        rms_features.append(rms)

        spectral_centroid=spectral_centroid_extractor(i)
        sc_features.append(spectral_centroid)

        spectral_bandwidth=spectral_bandwidth_extractor(i)
        sb_features.append(spectral_bandwidth)

        spectral_contrast=spectral_contrast_extractor(i)
        sco_features.append(spectral_contrast)

        poly=polynomial_extractor(i)
        poly_features.append(poly)
    extracted_features_df=pd.DataFrame([mfcc_features,zero_features,rms_features,sc_features,sb_features,sco_features,poly_features, labels])
    extracted_features_df=extracted_features_df.T
    extracted_features_df.columns=['mfcc','zero crossing rate','root mean square','spectral centroid','spectral bandwidth','spectral contrast','polynomial', 'labels']
    return extracted_features_df

In [45]:
df_train =feature_extractor(train_dataset)
df_test =feature_extractor(test_dataset)

labels_train = df_train.pop('labels')
labels_test = df_test.pop('labels')

df_train = df_train.applymap(lambda x: np.median(x))
df_test = df_test.applymap(lambda x: np.median(x))


NameError: name 'pd' is not defined

In [ ]:
df_train.head()

In [32]:
x=np.array(df_train).tolist()
x=np.array(x)
x_t=np.array(df_test).tolist()
x_t=np.array(x_t)

In [33]:
x.shape,x_t.shape

((2370, 7), (1101, 7))

In [34]:
from sklearn.preprocessing import StandardScaler, Normalizer
x=Normalizer().fit_transform(x)
x_t=Normalizer().fit_transform(x_t)
x=StandardScaler().fit_transform(x)
x_t=StandardScaler().fit_transform(x_t)

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(7, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, 16),
            nn.ELU(),
            nn.Linear(16, 8),
            nn.ELU(),
            nn.Linear(8, 4),
            nn.ELU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ELU(),
            nn.Linear(8, 16),
            nn.ELU(),
            nn.Linear(16, 32),
            nn.ELU(),
            nn.Linear(32, 64),
            nn.ELU(),
            nn.Linear(64, 7),
            nn.ELU()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Assume x and x_t are NumPy arrays with shape [num_samples, 7]
x_tensor = torch.tensor(x, dtype=torch.float32)
x_t_tensor = torch.tensor(x_t, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(x_tensor, x_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_t_tensor, x_t_tensor), batch_size=32, shuffle=False)

# Model, optimizer, loss
model = Autoencoder()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.05, patience=2, verbose=True)


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [37]:
num_epochs = 150

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch_x, _ in val_loader:
            output = model(batch_x)
            loss = criterion(output, batch_x)
            val_losses.append(loss.item())
            all_preds.append(output.numpy())
            all_targets.append(batch_x.numpy())

    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses)
    
    # Optional metrics
    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    mae = mean_absolute_error(targets, preds)
    try:
        msle = np.mean((np.log1p(preds) - np.log1p(targets))**2)
    except:
        msle = float("nan")

    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - MAE: {mae:.4f} - MSLE: {msle:.4f}")

    # Adjust learning rate
    scheduler.step(avg_val_loss)

Epoch 1/150 - Train Loss: 0.6914 - Val Loss: 0.5384 - MAE: 0.4287 - MSLE: nan
Epoch 2/150 - Train Loss: 0.4356 - Val Loss: 0.4633 - MAE: 0.3692 - MSLE: nan
Epoch 3/150 - Train Loss: 0.3920 - Val Loss: 0.4036 - MAE: 0.3545 - MSLE: nan
Epoch 4/150 - Train Loss: 0.2892 - Val Loss: 0.3107 - MAE: 0.3066 - MSLE: nan
Epoch 5/150 - Train Loss: 0.2257 - Val Loss: 0.2293 - MAE: 0.3089 - MSLE: nan
Epoch 6/150 - Train Loss: 0.1546 - Val Loss: 0.2149 - MAE: 0.3004 - MSLE: nan
Epoch 7/150 - Train Loss: 0.1449 - Val Loss: 0.2154 - MAE: 0.2954 - MSLE: nan
Epoch 8/150 - Train Loss: 0.1535 - Val Loss: 0.2007 - MAE: 0.2891 - MSLE: nan
Epoch 9/150 - Train Loss: 0.1432 - Val Loss: 0.1932 - MAE: 0.2858 - MSLE: nan
Epoch 10/150 - Train Loss: 0.1404 - Val Loss: 0.1887 - MAE: 0.2807 - MSLE: nan
Epoch 11/150 - Train Loss: 0.1349 - Val Loss: 0.1868 - MAE: 0.2785 - MSLE: nan
Epoch 12/150 - Train Loss: 0.1334 - Val Loss: 0.1857 - MAE: 0.2755 - MSLE: nan
Epoch 13/150 - Train Loss: 0.1323 - Val Loss: 0.1838 - MAE: 0

In [38]:
def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).reshape(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors

In [39]:
print(x_t)

[[-1.09543353e+00 -4.40873624e-02 -4.00205162e-01 ... -5.97088937e-02
  -3.22065224e-01 -3.40595958e-02]
 [ 3.39133205e-02  3.21384005e-01  4.20272881e-02 ...  2.78266240e-01
   2.45903808e-01 -3.25263782e-02]
 [-7.01452568e-01 -4.95744877e-01  4.81584445e-01 ... -2.90413062e-02
  -5.18737568e-01 -3.47068619e-02]
 ...
 [-1.61240167e-01  4.05362294e-01 -6.89972225e-01 ... -8.51176231e-01
  -3.18775645e-03 -4.19823684e-02]
 [-7.33866872e-01 -2.12633655e-01  2.58048234e-01 ...  3.15996533e-02
  -9.67688684e-01 -3.21677162e-02]
 [-8.61472384e-01 -2.03337606e+00  2.87323674e+00 ...  2.22495964e+00
   3.49035683e+00 -1.44916234e-02]]


In [ ]:
from sklearn.metrics import roc_auc_score
errors = compute_reconstruction_errors(model, val_loader)
roc_auc_scores = roc_auc_score(test_dataset.labels.cpu(), errors)
print(f"ROC AUC Score test: {roc_auc_scores}")
